# ECE1512 Project A - Build an ABMIL training pipeline in PyTorch

This notebook uses pre-extracted whole-slide histopathology features and guides you through a complete PyTorch classification pipeline. The goal is not merely to run ABMIL; it is to understand the code path from data loading to optimization, validation, checkpoint selection, and final testing.

Please complete every cell marked `TODO`. Dataset engineering and metric boilerplate are provided so you can focus on the machine-learning training workflow.

This project requires basic familiarity with Python object-oriented programming (OOP). If you have limited experience with Python OOP, please review the provided introductory video before starting the project. 

[PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/introyt1_tutorial.html)

[Python Object-Oriented Programming ](https://www.youtube.com/watch?v=JeznW_7DlB0)

**Research extension:** Propose two research ideas and implement and evaluate one selected idea. Report ABMIL baseline and selected-extension results on **CAMELYON16, CAMELYON17, and BRACS**, using the same features and train/validation/test splits within each dataset. 


## Learning objectives

After this project you should be able to explain and implement:

1. `Dataset` vs. `DataLoader`, batching, shuffling.
2. `nn.Module`, trainable parameters, and `forward()`.
3. A classification loss and optimizer.
4. The training sequence `zero_grad -> forward -> loss -> backward -> step`.
5. The difference between `model.train()` and `model.eval()`.

## Step 0. Download Pre-extracted WSI Features & Select the Project Directory
Download the required feature file(s) from Zenodo or Hugging Face:

https://zenodo.org/records/23130543

https://huggingface.co/datasets/LeonYe/ECE1512-WSI-Feature


Place the HDF5 file under the path expected by your YAML configuration in the `config/`, or edit data_dir in the YAML file.


In [ ]:
!pip install -q torchmetrics h5py pandas scikit-learn pyyaml matplotlib openslide-python openslide-bin


In [ ]:
import os, sys, copy, time
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import collections
import pickle
import pandas as pd

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import utils
from utils import TODO

If you are using Google Colab with Drive, mount Drive first and change `PROJECT_DIR`.

In [ ]:
# # Optional in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_DIR = './'
DATASET = 'C16'  # Choose: 'C16', 'C17', or 'BRACS'

CONFIG_FILES = {
    'C16': 'camelyon16_medical_ssl_config.yml',
    'C17': 'camelyon17_medical_ssl_config.yml',
    'BRACS': 'bracs_medical_ssl_config.yml',
}
CONFIG_PATH = os.path.join(PROJECT_DIR, 'config', CONFIG_FILES[DATASET])

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Project:', os.path.abspath(PROJECT_DIR))
print('Config :', CONFIG_PATH)

from wsi_visualization import visualize_wsi_attention


In [ ]:
from dataloader import build_HDF5_feat_dataset, mil_collate_fn
from utils import load_config, seed_everything, infer_feature_dim, classification_metrics

conf = load_config(CONFIG_PATH)
seed_everything(int(conf.seed))
print(conf)

if os.path.exists(conf.data_dir):
    print('Feature file exists:', os.path.exists(conf.data_dir), conf.data_dir)
else:
    print('\nDownload the required feature file from: https://zenodo.org/records/23130543')
    print('Then update data_dir in:', CONFIG_PATH)
    

# Step 1 - Dataset and DataLoader

`build_HDF5_feat_dataset` is provided. It creates train/validation/test `Dataset` objects without loading all slide features into memory at once.


Complete the three `DataLoader` definitions. Think about which split should be shuffled.

[Download WSI datasets](https://zenodo.org/records/23130543)

[Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)


[PyTorch DataLoader](https://docs.pytorch.org/docs/2.14/data.html#torch.utils.data.DataLoader)


[PyTorch Datasets](https://docs.pytorch.org/vision/stable/datasets.html)


In [ ]:
train_data, val_data, test_data = build_HDF5_feat_dataset(conf.data_dir, conf)
conf.D_feat = infer_feature_dim(train_data)
print('Split sizes: Train {}, Validation {}, Test {};'.format(len(train_data), len(val_data), len(test_data)))
print("Feature Shape: {};".format(tuple(train_data[1]['input'].shape)))
print("Label: {}; \nSlide ID: {}.".format(train_data[1]['label'], train_data[1]['slide_id']))

In [ ]:
batch_size=conf.B
num_workers=int(conf.n_worker)
pin_memory=bool(conf.pin_memory) and torch.cuda.is_available()
collate_fn=mil_collate_fn
print("DataLoader args: Batch size {}; \n \
      # CPUs {};  \n \
      pin_memory {};  \n \
      collate_fn {}".format(batch_size, num_workers, pin_memory, collate_fn))

In [ ]:
# TODO 1A: create the training DataLoader. Training data should be shuffled.
train_loader = TODO(dataset=train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_fn, pin_memory=pin_memory)

# TODO 1B: create validation and test DataLoaders. Do not shuffle them.
val_loader = TODO(dataset=val_data, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_fn, pin_memory=pin_memory)
test_loader = TODO(dataset=test_data, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_fn, pin_memory=pin_memory)


In [ ]:
# Inspect one MIL batch.
batch = next(iter(train_loader))
print('input:', tuple(batch['input'].shape), batch['input'].dtype)
print('label:', tuple(batch['label'].shape), batch['label'].dtype)
print('slide IDs:', batch['slide_id'])

**Questions for your report:**

- What is the difference between a `Dataset` and a `DataLoader`?
- Which Dataloader should use `shuffle=True`? Why?
- For a batch with input shape `[B, N, D]`, what do `B`, `N`, and `D` mean in this project?


# Step 2 - Implement the ABMIL model

For each patch feature $h_i$, the gated attention score follows Ilse et al. (2018):

$$
a_i = w^T\left[\tanh(Vh_i) \odot \sigma(Uh_i)\right].
$$

The attention logits are normalized over instances:

$$
\alpha_i = \frac{\exp(a_i)}{\sum_j \exp(a_j)},
$$

and the slide representation is

$$
z = \sum_i \alpha_i h_i.
$$

You will implement the learnable layers and the key `forward()` operations.

[Custom nn Modules](https://docs.pytorch.org/tutorials/beginner/examples_nn/polynomial_module.html)

[nn.Linear](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Linear.html)

[nn.modules.dropout](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Dropout.html)

[nn.ReLU](https://docs.pytorch.org/docs/2.14/generated/torch.nn.ReLU.html)

[torch.nn.functional.softmax (F.softmax)](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.softmax.html)

[torch.bmm](https://docs.pytorch.org/docs/2.14/generated/torch.bmm.html)

In [ ]:
class Classifier1FC(nn.Module):
    def __init__(self, n_channels, n_classes, droprate=0.0):
        super().__init__()
        self.dropout = nn.Dropout(droprate) if droprate > 0 else nn.Identity()
        # TODO 2A: one linear layer from n_channels to n_classes.
        self.fc = nn.Linear(in_features=n_channels, out_features=n_classes)

    def forward(self, x):
        # TODO 2B: apply dropout, then the classifier.
        x = self.dropout(x)                     # [B, n_channels]
        x = self.fc(x)                          # [B, n_classes]
        return x


class DimReduction(nn.Module):
    def __init__(self, n_channels, m_dim):
        super().__init__()
        # TODO 2C: project patch features to m_dim and use ReLU.
        self.fc = nn.Linear(in_features=n_channels, out_features=m_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        # TODO 2D: linear projection followed by ReLU.
        x = self.fc(x)
        x = self.relu(x)
        return x


class AttentionGated(nn.Module):
    def __init__(self, L, D, C=1):
        super().__init__()
        self.attention_V = nn.Sequential(nn.Linear(L, D), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(L, D), nn.Sigmoid())
        self.attention_weights_w = nn.Linear(D, C)

    def forward(self, x):
        # x: [B, N, L]
        # TODO 2E: implement the two gated branches.
        a_v = self.attention_V(x)
        a_u = self.attention_U(x)

        a = self.attention_weights_w(a_v * a_u)
        return a.transpose(1, 2)


class ABMIL(nn.Module):
    def __init__(self, conf, multiClassAttn=False):
        super().__init__()
        if multiClassAttn:
            C = conf.n_class
        else:
            C = 1
        self.dimreduction = DimReduction(conf.D_feat, conf.D_inner)
        self.attention = AttentionGated(conf.D_inner, conf.D_attn, C=C)
        self.classifier = Classifier1FC(C*conf.D_inner, conf.n_class, conf.droprate)

    def forward(self, x, return_attention=False):
        # x: [B, N, D_feat]
        med_feat = self.dimreduction(x)       # [B, N, D_inner]
        a_logits = self.attention(med_feat)   # [B, C, N]

        # TODO 2G: normalize attention logits over the instance dimension.
        attention = F.softmax(a_logits, dim=-1)     # [B, C, N]
        
        # TODO 2H: weighted aggregation. torch.bmm is useful here.
        bag_feat = torch.bmm(attention, med_feat).flatten(1)   # [B, C*D_inner]

        # TODO 2I: classify the slide representation.
        logits = self.classifier(bag_feat)    # [B, n_class]

        if return_attention:
            return logits, attention
        return logits


**Sanity Check**

Before running the full training pipeline, check the tensor shapes and attention normalization.


In [ ]:
model_cpu = ABMIL(conf).eval()

N = torch.randint(low=0,high=100,size=(1,)).item()
x_demo = torch.randn(2, N, conf.D_feat)

with torch.no_grad():
    logits_demo, attn_demo = model_cpu(x_demo, return_attention=True)

print('logits:', logits_demo.shape)
print('attention:', attn_demo.shape)
print('attention sums:', attn_demo.sum(dim=-1))

assert logits_demo.shape == (2, int(conf.n_class))
assert attn_demo.shape == (2, 1, N)
torch.testing.assert_close(attn_demo.sum(dim=-1), torch.ones(2, 1))
print('Sanity check passed.')


**Questions:**

- Which tensors in your model contain trainable parameters?
- What properties do attention weights have after softmax?
- Could we replace softmax with another function?


# Step 3 - Device, loss, and optimizer

For multi-class slide classification, use cross-entropy as the baseline objective and AdamW as the baseline optimizer. Complete the code below.

[nn.CrossEntropyLoss](https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html)

[torch.optim.AdamW](https://docs.pytorch.org/docs/2.14/generated/torch.optim.AdamW.html)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DATASET == 'C16':
    conf.droprate = 0.0 # no dropout required for simple binary classification
model = ABMIL(conf).to(device)

# TODO 3A: define the multi-class classification loss.
criterion = nn.CrossEntropyLoss()   # default reduction='mean'

# TODO 3B: create AdamW and pass the model parameters, learning rate, and weight decay.
optimizer = torch.optim.AdamW(params=model.parameters(),
                              lr=conf.lr,
                              weight_decay=conf.wd,
                              fused=True)

print('Device:', device)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


# Step 4 - Complete one training epoch

The essential PyTorch optimization sequence is:

`model.train()` -> move tensors to device -> `zero_grad()` -> forward pass -> compute loss -> `backward()` -> `step()`.

Implement this sequence yourself. This is one of the main learning objectives of the project.

[Training with PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html)

[train() vs eval()](https://discuss.pytorch.org/t/model-train-and-model-eval-vs-model-and-model-eval/5744)

In [ ]:
def train_one_epoch(model, criterion, data_loader, optimizer, device):
    # TODO 4A: put the model in training mode.
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for data in data_loader:
        # TODO 4B: move input and labels to the selected device.
        bag = data["input"].to(device, dtype=torch.float32)
        labels = data["label"].to(device, dtype=torch.long)

        # TODO 4C: clear gradients from the previous iteration.
        optimizer.zero_grad()

        # TODO 4D: forward pass.
        logits = model(bag)

        # TODO 4E: compute the loss.
        loss = criterion(logits, labels)

        # TODO 4F: back-propagate gradients.
        loss.backward()

        # TODO 4G: update the model parameters.
        optimizer.step()

        batch_size = labels.shape[0]
        total_loss += loss.item() * batch_size # reduction='mean'
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    return {
        'loss': total_loss / total_examples,
        'acc': 100.0 * total_correct / total_examples,
    }


# Step 5 - Validation and test evaluation

Evaluation should not update the model. Complete the missing lines and use the provided metric helper for macro-AUROC and macro-F1.


In [ ]:
@torch.no_grad()
def evaluate(model, criterion, data_loader, device, conf):
    # TODO 5A: switch to evaluation mode.
    model.eval()

    total_loss = 0.0
    total_examples = 0
    probabilities_all = []
    labels_all = []

    for data in data_loader:
        # TODO 5B: move data to device.
        bag = data["input"].to(device, dtype=torch.float32)
        labels = data["label"].to(device, dtype=torch.long)

        # TODO 5C: forward pass and loss.
        logits = model(bag)
        loss = criterion(logits, labels)

        # TODO 5D: convert class logits into probabilities.
        probabilities = F.softmax(logits, dim=-1)

        total_loss += loss.item() * labels.shape[0] # reduction='mean'
        total_examples += labels.shape[0]
        probabilities_all.append(probabilities)
        labels_all.append(labels)

    probabilities_all = torch.cat(probabilities_all, dim=0)
    labels_all = torch.cat(labels_all, dim=0)
    metrics = classification_metrics(probabilities_all, labels_all, conf.n_class)
    metrics['loss'] = total_loss / total_examples
    return metrics


**Checkpoint questions:**

- Why are `model.eval()` and `torch.no_grad()` different concepts?
- What would happen if gradients were accumulated because `optimizer.zero_grad()` was omitted?
- Why must the test set not be used to choose the best epoch?


# Step 6 - Train, validate, and select the best checkpoint

Use the validation set to choose the best epoch. The test set must remain untouched until model selection is complete.

For debugging, set `NUM_EPOCHS = 2`. For your reported baseline, use the full configured epoch count unless instructed otherwise.


In [ ]:
def train(model: nn.Module,
          criterion: nn.modules.loss._Loss,
          optimizer: torch.optim.Optimizer,
          train_loader: DataLoader,
          val_loader: DataLoader,
          conf: utils.Struct,
          device: torch.device) -> utils.Result:
    NUM_EPOCHS = int(conf.train_epoch)  # You may temporarily set this to 2 while debugging.

    best_score = float('-inf')
    best_val_loss = float('inf')
    best_epoch = -1
    best_model_state = None
    history = []

    for epoch in range(NUM_EPOCHS):
        start = time.time()
        train_metrics = train_one_epoch(model, criterion, train_loader, optimizer, device)
        val_metrics = evaluate(model, criterion, val_loader, device, conf)

        # Baseline selection criterion: macro-AUROC + macro-F1.
        score = val_metrics['auc'] + val_metrics['f1']
        if score >= best_score:
            # secondary criterion: val loss
            val_loss = val_metrics['loss']
            if val_loss < best_val_loss:
                best_val_loss = val_loss
            if (score > best_score) or (val_loss == best_val_loss):
                best_score = score
                best_epoch = epoch
                best_model_state = copy.deepcopy(model.state_dict())
                
        row = {
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_acc': train_metrics['acc'],
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
            'val_auc': val_metrics['auc'],
            'val_f1': val_metrics['f1'],
        }
        history.append(row)
        print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
            f"train loss={row['train_loss']:.4f} acc={row['train_acc']:.2f} | "
            f"val loss={row['val_loss']:.4f} auc={row['val_auc']:.3f} f1={row['val_f1']:.3f} | "
            f"best={best_epoch+1} | {time.time()-start:.1f}s")

    assert best_model_state is not None
    return utils.Result(model_type=type(model), history=history, best_epoch=best_epoch, best_model_state=best_model_state)

result = train(model=model, criterion=criterion, optimizer=optimizer,
               train_loader=train_loader, val_loader=val_loader,
               conf=conf, device=device)

In [ ]:
# Plot learning curves.
def plot_learning_curves(history: list):
    epochs = [x['epoch'] + 1 for x in history]
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, [x['train_loss'] for x in history], label='train loss')
    plt.plot(epochs, [x['val_loss'] for x in history], label='validation loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

plot_learning_curves(history=result.history)

# Step 7 - Final held-out test evaluation

Only now, after the best checkpoint has been chosen from validation performance, load it and evaluate once on the test set.


In [ ]:
def test(model: nn.Module,
         best_epoch: int,
         best_model_state: collections.OrderedDict,
         criterion: nn.modules.loss._Loss,
         test_loader: DataLoader,
         conf: utils.Struct,
         device: torch.device):
    model.load_state_dict(best_model_state)
    test_metrics = evaluate(model, criterion, test_loader, device, conf)

    print('Best validation epoch:', best_epoch + 1)
    print('Final test metrics:')
    for key, value in test_metrics.items():
        print(f'  {key}: {value:.4f}')

test(model=model, best_epoch=result.best_epoch, best_model_state=result.best_model_state, criterion=criterion, test_loader=test_loader, conf=conf, device=device)

# Step 8 -  Inspect Attention on One Slide

Attention is useful for understanding which instances contribute strongly to the learned bag representation, but attention weights should not automatically be interpreted as causal explanations.


In [ ]:
one_batch = mil_collate_fn([test_data[0]])
model.eval()
with torch.no_grad():
    logits, attention = model(
        one_batch['input'].to(device, dtype=torch.float32),
        return_attention=True,
    )

print('Slide:', one_batch['slide_id'][0])
print('True class:', one_batch['label'].item())
print('Predicted class:', int(logits.argmax(dim=1)[0]))
print('Attention shape:', tuple(attention.shape))
print('Attention sum for first slide:', float(attention[0].sum()))


## Visualize Attention on Whole-Slide Images (WSIs)

Before running the attention visualization, please complete the following steps:

1. Visit the [ECE1512-WSI-Feature dataset on Hugging Face](https://huggingface.co/datasets/LeonYe/ECE1512-WSI-Feature)

2. Download the original WSIs `test_067.tif` and `test_068.tif` and the corresponding tumor annotation file `test_068.xml`.

3. Create a folder named `slide/` under your project directory and place the downloaded files in this folder.

Run this cell after training and final test evaluation, with `DATASET = 'C16'`. It restores `best_model_state` and plots the two slides in `PROJECT_DIR/slide`:

- **test_068 (tumor):** attention overlay and green contours from `test_068.xml`, including a zoom of the annotated region.
- **test_067 (normal):** attention overlay without annotation contours, including a zoom around the highest-attention patch.

The blue rectangle marks the zoom shown on the right. Colors show **attention percentiles within each slide**, not patch tumor probabilities; areas without extracted patches stay transparent. The displayed `P(tumor)` is the model's slide-level prediction.

HDF5 patch coordinates and XML contour coordinates are assumed to be expressed in level-0 `(x, y)`  coordinates; patch coordinates are treated as top-left corners. The function parameter `patch_size_level0=512` is an editable **assumption** for the patch footprint because these HDF5 files do not store patch-size metadata. Match it to the feature extractor's patch size for an exact footprint. All visualization functions are in `wsi_visualization.py`; the cell below only calls `visualize_wsi_attention`.

[OpenSlide Python](https://openslide.org/api/python/) reads a thumbnail and a pyramid-level crop rather than loading the full-resolution WSI into memory. If dependencies are missing, run `%pip install openslide-python openslide-bin` and restart the kernel if needed. Raw per-patch attention and coordinates are retained in `attention_visualizations`.


In [ ]:
if DATASET == 'C16':
    attention_visualizations = visualize_wsi_attention(model, conf, PROJECT_DIR, result.best_model_state)


# Step 9 - Research extension: propose two ideas, implement one of them

After completing the ABMIL baseline, propose **two distinct research ideas**. Select **one** for implementation and evaluation on **CAMELYON16 (C16), CAMELYON17 (C17), and BRACS**. You may explore explainability, more advanced model structures, improved objectives beyond cross-entropy, or another well-motivated direction.

**A negative result will not be penalized if the design choices, experiments, observations, and possible reasons for failure are well justified.** Report evidence relevant to your hypothesis: an explainability idea, for example, needs an explanation evaluation in addition to classification results.

Feel free to contact your TA to discuss your research ideas.

# Multi-Class Attention-Based Multiple Instance Learning (MC-ABMIL)

## Single Run

In [ ]:
MC_ABMIL = ABMIL(conf=conf, multiClassAttn=True).to(device)
optimizer = torch.optim.AdamW(params=MC_ABMIL.parameters(),
                              lr=conf.lr,
                              weight_decay=conf.wd,
                              fused=True)
MC_ABMIL_result = train(model=MC_ABMIL, criterion=criterion, optimizer=optimizer,
                        train_loader=train_loader, val_loader=val_loader,
                        conf=conf, device=device)

In [ ]:
plot_learning_curves(history=MC_ABMIL_result.history)
test(model=MC_ABMIL, best_epoch=MC_ABMIL_result.best_epoch, best_model_state=MC_ABMIL_result.best_model_state, criterion=criterion, test_loader=test_loader, conf=conf, device=device)

## Multi-Run

In [ ]:
def train_multirun(model_type: type[nn.Module],
                   model_args: dict,
                   criterion: nn.modules.loss._Loss,
                   train_loader: DataLoader,
                   val_loader: DataLoader,
                   device: torch.device,
                   seeds: list) -> dict[utils.Result]:
    conf = model_args['conf']
    results = {}
    for seed in seeds:
        utils.seed_everything(seed)
        model = model_type(**model_args).to(device)
        optimizer = torch.optim.AdamW(params=model.parameters(),
                                      lr=conf.lr,
                                      weight_decay=conf.wd,
                                      fused=True)
        results[seed] = train(model=model, criterion=criterion, optimizer=optimizer,
                              train_loader=train_loader, val_loader=val_loader,
                              conf=conf, device=device)
        results[seed].model_args = model_args
    return results

def evaluate_multirun(results: dict[utils.Result],
                      criterion: nn.modules.loss._Loss,
                      test_loader: DataLoader,
                      device: torch.device) -> tuple[dict, dict]:
    test_metrics = []
    for _, result in results.items():
        model = result.model_type(**result.model_args).to(device)
        model.load_state_dict(result.best_model_state)
        test_metrics.append(evaluate(model, criterion, test_loader, device, result.model_args['conf']))
    mean = pd.DataFrame(test_metrics).mean().to_dict()
    std = pd.DataFrame(test_metrics).std().to_dict()

    print('Final test metrics:')
    for key, value in mean.items():
        std_dev = std[key]
        print(f'  {key}: {value:.4f} ± {std_dev:.4f}')
    return mean, std

seeds = [conf.seed, 15, 12, 17, 73]

In [ ]:
ABMIL_results = train_multirun(model_type=ABMIL, model_args={'conf': conf, 'multiClassAttn': False},
                               criterion=criterion,
                               train_loader=train_loader, val_loader=val_loader,
                               device=device, seeds=seeds)
with open("ABMIL_results_" + DATASET + ".pkl", "wb") as f:
    pickle.dump(ABMIL_results, f)

In [ ]:
MC_ABMIL_results = train_multirun(model_type=ABMIL, model_args={'conf': conf, 'multiClassAttn': True},
                                  criterion=criterion,
                                  train_loader=train_loader, val_loader=val_loader,
                                  device=device, seeds=seeds)
with open("MC_ABMIL_results_" + DATASET + ".pkl", "wb") as f:
    pickle.dump(MC_ABMIL_results, f)

In [ ]:
with open("ABMIL_results_" + DATASET + ".pkl", "rb") as f:
    ABMIL_results = pickle.load(f)
with open("MC_ABMIL_results_" + DATASET + ".pkl", "rb") as f:
    MC_ABMIL_results = pickle.load(f)
utils.visualise_stats(results=ABMIL_results)
utils.visualise_stats(results=MC_ABMIL_results)
evaluate_multirun(results=ABMIL_results, criterion=criterion, test_loader=test_loader, device=device);
evaluate_multirun(results=MC_ABMIL_results, criterion=criterion, test_loader=test_loader, device=device);

# Knowledge-Aware Attention for Multi-class Correlated MIL
Adapted from [Dynamic Graph Representation with Knowledge-aware Attention for Histopathology Whole Slide Image Analysis](https://arxiv.org/abs/2403.07719)

In [ ]:
# adapted from https://github.com/WonderLandxD/WiKG/tree/main

class WiKG(nn.Module):
    def __init__(self, conf: utils.Struct, chunk_size=None, multiClassAttn=False):
        super().__init__()
        self.chunk_size = chunk_size

        self.fc = nn.Sequential(nn.Linear(conf.D_feat, conf.D_inner), nn.LeakyReLU())
        
        self.W_head = nn.Linear(conf.D_inner, conf.D_inner)
        self.W_tail = nn.Linear(conf.D_inner, conf.D_inner)

        self.scale = conf.D_inner ** -0.5
        self.topk = conf.k

        self.linear1 = nn.Linear(conf.D_inner, conf.D_inner)
        self.linear2 = nn.Linear(conf.D_inner, conf.D_inner)

        if (multiClassAttn
            and conf.n_class > 2):  # binary classification can be reduced to single classification
            C = conf.n_class
        else:
            C = 1
        self.activation = nn.LeakyReLU()
        self.attn = AttentionGated(conf.D_inner, conf.D_attn, C=C)
        self.classifier = Classifier1FC(C*conf.D_inner, conf.n_class, conf.droprate)

    @torch.no_grad()    # gradients do not need to flow through indices
    def _topk_neighbors(self, e_h, e_t):
        """e_h, e_t: [B, N, D] -> top-k indices: [B, N, k]"""
        q = e_h * self.scale
        e_t_T = e_t.transpose(-2, -1)   # [B, D, N]

        # chunk to keep within memory limits
        idx = []
        chunk_size = self.chunk_size or e_h.size(1)
        k = min(self.topk, e_h.size(1))
        for s in range(0, q.size(1), chunk_size):
            logit = q[:, s:s + chunk_size] @ e_t_T      # [B, c, N], c: chunk size
            idx.append(logit.topk(k, dim=-1).indices)   # [B, c, k]
        return torch.cat(idx, dim=1)
    
    def forward(self, x, return_attention=False):   # x: [B, N, D_feat]
        x = self.fc(x)  # [B, N, D]

        e_h = self.W_head(x)    # head embedding [B, N, D]
        e_t = self.W_tail(x)    # tail embedding [B, N, D]

        # 1. neighbour (top-k tails) search (no grad)
        topk_index = self._topk_neighbors(e_h, e_t) # [B, N, k]

        # 2. neighbour set construction
        batch_indices = torch.arange(e_t.size(0), device=e_t.device).view(-1, 1, 1) # [B, 1, 1]
        Nb_h = e_t[batch_indices, topk_index, :]                                    # [B, N, k, D]
        
        # 3. edge embedding
        topk_weight = torch.einsum('bnd,bnkd->bnk', e_h * self.scale, Nb_h)         # [B, N, k]
        topk_prob = F.softmax(topk_weight, dim=2)                                                   # [B, N, k]
        eh_r = topk_prob.unsqueeze(-1) * Nb_h + (1 - topk_prob).unsqueeze(-1) * e_h.unsqueeze(2)    # [B, N, k, D]
        
        # 4. gated knowledge attention
        gate = torch.tanh(e_h.unsqueeze(2) + eh_r)              # [B, N, k, D]
        ka_weight = (Nb_h * gate).sum(dim=-1)                   # [B, N, k]
        ka_prob = F.softmax(ka_weight, dim=2).unsqueeze(dim=2)  # [B, N, 1, k]
        # 4.1 aggregate neighbour information
        e_Nh = torch.matmul(ka_prob, Nb_h).squeeze(dim=2)       # [B, N, D]

        # 5. fused head embeddings
        sum_h = self.activation(self.linear1(e_h + e_Nh))   # [B, N, D]
        bi_h = self.activation(self.linear2(e_h * e_Nh))    # [B, N, D]
        h = sum_h + bi_h                                    # [B, N, D]
            
        # 6. attention pooling
        attention = F.softmax(self.attn(h), dim=-1)     # [B, C, N]
        bag_feat = torch.bmm(attention, h).flatten(1)   # [B, C*D]
        logits = self.classifier(bag_feat)              # [B, C]
        
        if return_attention:
            return logits, attention
        return logits

In [ ]:
chunk_size = 1024   # adjust to your memory limits
WiKG_conf = copy.deepcopy(conf)
WiKG_conf.k = 6
WiKG_model = WiKG(conf=WiKG_conf, chunk_size=chunk_size, multiClassAttn=True).to(device)
WiKG_criterion = nn.CrossEntropyLoss()   # default reduction='mean'
WiKG_optimizer = torch.optim.AdamW(params=WiKG_model.parameters(),
                                   lr=WiKG_conf.lr,
                                   weight_decay=WiKG_conf.wd,
                                   fused=True)

WiKG_result = train(model=WiKG_model, criterion=WiKG_criterion, optimizer=WiKG_optimizer,
                    train_loader=train_loader, val_loader=val_loader,
                    conf=conf, device=device)

In [ ]:
plot_learning_curves(history=WiKG_result.history)
test(model=WiKG_model, best_epoch=WiKG_result.best_epoch, best_model_state=WiKG_result.best_model_state, criterion=WiKG_criterion, test_loader=test_loader, conf=conf, device=device)

In [ ]:
# Attention visualisation
if DATASET == 'C16':
    WiKG_av = visualize_wsi_attention(WiKG_model, WiKG_conf, PROJECT_DIR, WiKG_result.best_model_state)